# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a git pull and merge your changes as needed</a>. Check out the GitHub guide for instructions. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

### Importing the libraries

In [ ]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter
from google import genai
from openai import OpenAI
from IPython.display import display, update_display, Markdown

In [ ]:
## loading the api keys

load_dotenv(override=True)

open_router_api_key = os.getenv('OPENROUTER_API_KEY')
gemini_api_key = os.getenv('GOOGLE_API_KEY')
deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')

if (open_router_api_key and open_router_api_key.startswith('sk-or')):
    print('API key found and good so far')
else:
    print('Api Key not found')

if gemini_api_key:
    print(f"Google API Key exists and begins {gemini_api_key[:2]}")
else:
    print("Google API Key not set")

if deepseek_api_key:
    print(f"DeepSeek API Key exists and begins {deepseek_api_key[:3]}")
else:
    print("DeepSeek API Key not set (and this is optional)")



In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints

openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# deepseek_url = "https://api.deepseek.com"
open_router_url = "https://openrouter.ai/api/v1"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_url)
# deepseek = OpenAI(api_key=deepseek_api_key, base_url=deepseek_url)
open_router = OpenAI(api_key=open_router_api_key, base_url=open_router_url)
ollama = OpenAI(api_key='ollama', base_url=ollama_url)



### OpenRouter Free Models

In [ ]:
gemini_model_gemma_4_31b = 'google/gemma-4-31b-it:free'

gpt_oss_model = 'openai/gpt-oss-120b:free'

inclusionai_model_ring_2_1t = 'inclusionai/ring-2.6-1t:free'

In [ ]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [ ]:
def call_openai(client_obj, model, messages):
    response = client_obj.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content


In [ ]:
result = call_openai(gemini, 'gemini-2.5-flash-lite', tell_a_joke)
display(Markdown(result))

In [ ]:
result = call_openai(open_router, gpt_oss_model, tell_a_joke)
display(Markdown(result))

## Training vs Inference time scaling

In [ ]:
easy_puzzle = [
    {"role": "user", "content": 
    "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."}
]


In [ ]:
result = call_openai(gemini, 'gemini-2.5-flash-lite', easy_puzzle)
display(Markdown(result))

In [ ]:
result = call_openai(open_router, gpt_oss_model, easy_puzzle)
result

## Testing out with hard problems

In [ ]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [ ]:
result = call_openai(open_router, gpt_oss_model, hard_puzzle)
display(Markdown(result))

In [ ]:
result = call_openai(open_router, inclusionai_model_ring_2_1t, hard_puzzle)
display(Markdown(result))

In [ ]:
result = call_openai(gemini, 'gemini-2.5-flash-lite', hard_puzzle)
display(Markdown(result))

### Gemini Endpoints

In [ ]:
gemini = genai.Client()

def call_gemini():

    response = gemini.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=easy_puzzle[0]["content"]
    )

    return response.text


In [ ]:
call_gemini()

### Defining Open Source Models from OpenRouter offerings

## OpenRouter
### 1. Gemini call
### 2. Inclusion AI call

In [ ]:
openrouter = OpenRouter(api_key=open_router_api_key)

def call_openrouter(model_name):
    messages = [
        {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"}
    ]

    stream = openrouter.chat.send(
        model = model_name,
        messages=messages,
        stream = True
    )

    response = ''
    display_handle = display(Markdown(''), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# calling the gemini gemma 4 31b model
call_openrouter(gemini_model_gemma_4_31b)

In [ ]:
# calling the inclusionai_model_ring_2.6 1t model

call_openrouter(inclusionai_model_ring_2_1t)